# Privacy & Data Handling [Security - Module 04]

> **MLCourse - Agentic AI - Production Security**

Agents routinely process text and data that contain personally
identifiable information (PII) -- names, emails, phone numbers, addresses,
account IDs. Sending raw PII to a model, logging it, or caching it can
violate privacy expectations and regulations. This module covers detecting
and redacting PII before it reaches an LLM, plus the general principles of
data minimization and local-first privacy. No API key required.

### What you will learn

1. What counts as PII and why it is risky in agent pipelines.
2. Detecting PII with deterministic pattern matching.
3. Redaction: masking PII before the model sees it.
4. When PII must be kept (vs. fully removed).
5. Data minimization principles.
6. Local-first privacy (why running models locally can help).
7. Composing a privacy guard into the agent pipeline.

### Key takeaways

- Redact PII before sending to a model, not after.
- Detect using patterns, then classify by type.
- Masking vs. removal depends on whether the task needs the value.
- Log redacted data, not raw data.
- Never send secrets or PII to logs, caches, or external providers.

### Setup: imports, environment


In [ ]:
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives inside the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)
print("Module 04: Privacy & Data Handling")
print(f"Track root: {TRACK}")


### 1. What Counts as PII

PII is any information that can identify an individual. Common examples
in agent pipelines:

- Emails, phone numbers, addresses.
- Names, usernames, account IDs.
- SSN / national IDs, passport numbers.
- Financial details (cards, IBAN).
- Behavioral or preference data tied to an individual.

Sending PII to a third-party model, storing it in a shared cache, or
logging it creates privacy and compliance risk.

### PII categories and patterns


In [ ]:
PII_PATTERNS = {
    "email":    r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    "phone":    r"(?:\+?\d{1,3}[-.\s]?)?\(?\d{2,4}\)?[-.\s]?\d{3,4}[-.\s]?\d{3,4}",
    "ssn":      r"\b\d{3}-\d{2}-\d{4}\b",
    "credit":   r"\b(?:\d[ -]?){13,16}\b",
    "address":  r"\b\d{1,5}\s+[A-Za-z0-9]+(?:\s+[A-Za-z0-9]+)*\s+(?:St|Ave|Rd|Blvd|Ln|Dr)\.?\b",
}

print("=== PII categories tracked ===\n")
for name in PII_PATTERNS:
    print(f"  - {name}")


### 2. Detecting PII

Detection is the first step: scan the text and report which categories and
how many instances were found. This is deterministic and cheap, suitable
for a pre-model guard.

### PII detector


In [ ]:
class PIIDetector:
    def __init__(self, patterns: dict = None):
        self.patterns = patterns or PII_PATTERNS
        self.compiled = {k: re.compile(v, re.IGNORECASE)
                         for k, v in self.patterns.items()}
    def find(self, text: str) -> List[Tuple[str, str]]:
        """Return list of (category, match)."""
        hits = []
        for cat, pat in self.compiled.items():
            for m in pat.finditer(text):
                hits.append((cat, m.group(0)))
        return hits

detector = PIIDetector()
sample = "Contact jane@example.com or 555-123-4567. SSN 123-45-6789."
print("=== Detection ===\n")
hits = detector.find(sample)
for cat, value in hits:
    print(f"  {cat:8s}: {value}")


### 3. Redaction: Masking PII

Redaction replaces PII with a placeholder, keeping the structural shape so
the model can still reason about the sentence without seeing the secret.
Masking is useful when the task needs the text but not the sensitive value.

Example: "email jane@example.com" -> "email <EMAIL>".

Each category gets its own token so downstream logic can unfuzz if needed
(but by default the redacted text is what is sent to the model).

### Redaction


In [ ]:
class Redactor:
    def __init__(self, detector: PIIDetector):
        self.detector = detector
    def redact(self, text: str) -> str:
        redacted = text
        for cat, _ in self.detector.find(text):
            pat = self.detector.compiled[cat]
            redacted = pat.sub(f"<{cat.upper()}>", redacted)
        return redacted
    def log_html(self, text: str) -> str:
        """Return a safe, redacted version for logs."""
        return self.redact(text)

redactor = Redactor(detector)
raw = "Order from jane@example.com, call 555-123-4567, SSN 123-45-6789."
print("=== Redaction ===\n")
print("RAW     :", raw)
print("REDACTED:", redactor.redact(raw))


### 4. Masking vs. Removal

Two strategies:

- **Masking** (replace with token): keeps sentence structure; the model can
  still answer generic questions about the message. Risk: the token is
  still there; the volume of PII matters.
- **Removal** (delete entirely): safest, but can break grammar or remove
  information the task needs (e.g., "the customer in New York" loses the
  location if fully removed).

Choose based on whether the task needs the value. For most agent tasks,
masking is a good default; for the highest-sensitivity data, remove it.

### Mask vs remove comparison


In [ ]:
text = "Please call John at 555-1234 about the delivery to 100 Main St."
print("=== Mask vs Remove ===\n")
print("RAW      :", text)
print("MASKED   :", redactor.redact(text))
# Removal: drop PII tokens entirely
removed = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|\+?\d[\d\s-]{7,}", "", text)
removed = re.sub(r"\s+", " ", removed).strip()
print("REMOVED  :", removed)


### 5. Data Minimization

Data minimization means sending the **least** data needed for the task:

- Do not send entire documents when a summary will do.
- Do not send an entire user record when the model only needs one field.
- Strip metadata and columns the task does not use.
- Prefer aggregate / derived values over raw PII.

Fewer PII tokens in the prompt = less leakage surface, smaller cost,
weaker incentive for an attacker to extract.

### Minimization example


In [ ]:
full_record = {
    "name": "Jane Doe", "email": "jane@example.com", "phone": "555-0100",
    "order_total": 120.0, "shipping_city": "Austin", "notes": "Call after 5pm",
}
print("=== Data minimization ===\n")
print("Full record fields:", list(full_record.keys()))
minimal = {k: full_record[k] for k in ("order_total", "shipping_city")}
print("Send only what is needed:", minimal)
print("PII-aware: redact name/email/phone/notes before any call.")


### 6. Local-First Privacy

Running the model locally (e.g., Ollama) keeps the data on your machine,
which is a strong privacy position: nothing leaves your control. This
complements redaction -- local-first limits *where* data goes, while
redaction limits *what* is in a given payload regardless of destination.

### Local vs cloud privacy comparison


In [ ]:
print("=== Local-first vs cloud ===\n")
compare = [
    ("Local (Ollama)",      "Data stays on machine", "needs GPU/RAM"),
    ("Cloud provider",      "No local compute",      "data leaves machine"),
    ("Local + redaction",   "Best of both",          "redaction logic needed"),
]
for name, pro, con in compare:
    print(f"  {name:20s} pro={pro:30s} con={con}")


### 7. Composing the Privacy Guard

The privacy guard runs as the very first step, before any model or tool:

1. Detect PII in the input.
2. Redact (or reject) before sending to the model.
3. Redact before logging anything.
4. Do not cache the raw input; cache only redacted text.

This is independent of, and complementary to, the injection detector and
guardrail pipeline from the earlier modules.

### Privacy pipeline


In [ ]:
class PrivacyGuard:
    def __init__(self):
        self.detector = PIIDetector()
        self.redactor = Redactor(self.detector)
    def sanitize(self, text: str) -> dict:
        hits = self.detector.find(text)
        return {
            "pii_count": len(hits),
            "categories": sorted({c for c, _ in hits}),
            "safe_text": self.redactor.redact(text),
        }

privacy = PrivacyGuard()
msg = "Hi, it's Jane (jane@example.com, 555-0100). Reset my password."
r = privacy.sanitize(msg)
print("=== Privacy guard ===\n")
print("PII count   :", r["pii_count"])
print("Categories  :", r["categories"])
print("Safe to log :", r["safe_text"])


### 8. Logging and Caching Rules

Privacy extends to wherever the data travels:

- **Logs**: log only redacted text.
- **Caches**: never cache raw PII (ties into Module 03).
- **Retrieval context**: redact retrieved documents containing PII before
  injecting them into the prompt.
- **External providers**: redact before sending to a cloud model.

### End-to-end privacy + logging example


In [ ]:
def handle_user(user_input: str):
    infos = privacy.sanitize(user_input)
    # pretend we call a model with the safe text
    safe = infos["safe_text"]
    log_line = f"[LOG] pii={infos['pii_count']} safe='{safe}'"
    return infos, log_line

sample_s = "My email is bob@site.com, send me the report."
info, log = handle_user(sample_s)
print("=== Logging example ===\n")
print("Log (redacted):", log)
print("Never log raw :", sample_s)


### Summary

- Detect PII deterministically before the model sees text.
- Mask PII with category tokens; remove it for the most sensitive data.
- Minimize the data you send in the first place.
- Local-first inference keeps data on your machine.
- Redact before logging, caching, or sending to external providers.
- Compose the privacy guard before the injection and guardrail layers.

### Final summary


In [ ]:
print("=== Module 04 Summary ===\n")
summary = [
    "PII detect: pattern-based, deterministic.",
    "Redact: mask PII -> <EMAIL>, <PHONE>, <SSN> tokens.",
    "Minimize: send only what the task needs.",
    "Local-first: keep data off external providers.",
    "Apply before: logs, caches, external calls, prompt building.",
]
for s in summary:
    print("  -", s)
